# MLP Training on `micro_mobility_training_data_2025.csv`

This notebook trains an `MLPRegressor` with the same chronological split logic as your baseline (last 7 days holdout).


## Colab Setup

- This notebook auto-detects Colab and mounts Google Drive.
- Update `COLAB_PROJECT_ROOT` if your Drive path is different.
- Keep your folder structure the same as local project.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [2]:
# Paths and config
# Set this to your repo folder under Google Drive (same structure as local project)
COLAB_PROJECT_ROOT = Path('/content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction')

IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = COLAB_PROJECT_ROOT
else:
    # local fallback
    PROJECT_ROOT = Path.cwd().resolve().parents[1]

DATA_PATH = PROJECT_ROOT / 'data' / 'proceed' / 'micro_mobility_training_data_2025.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'model_training' / 'mlp' / 'v1'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_DAYS = 7
RANDOM_STATE = 42

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_PATH:', DATA_PATH)
print('ARTIFACT_DIR:', ARTIFACT_DIR)


Mounted at /content/drive
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction
DATA_PATH: /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/data/proceed/micro_mobility_training_data_2025.csv
ARTIFACT_DIR: /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/mlp/v1


In [3]:
# Load data
cols = [
    'station', 'date', 'hour', 'net_demand',
    'lat', 'lng', 'hour_sin', 'hour_cos', 'day_of_week', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h'
]

df = pd.read_csv(DATA_PATH, usecols=cols)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'station', 'net_demand']).copy()

# Stable station encoding for MLP
stations_sorted = sorted(df['station'].dropna().unique().tolist())
station_to_id = {s: i for i, s in enumerate(stations_sorted)}
df['station_id'] = df['station'].map(station_to_id).astype('int32')

# Chronological split
cutoff_date = df['date'].max().normalize() - pd.Timedelta(days=HOLDOUT_DAYS)
train_mask = df['date'] <= cutoff_date
test_mask = df['date'] > cutoff_date

feature_cols = [
    'station_id', 'hour', 'lat', 'lng', 'hour_sin', 'hour_cos',
    'day_of_week', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h'
]

X_train = df.loc[train_mask, feature_cols].astype('float32')
y_train = df.loc[train_mask, 'net_demand'].astype('float32')
X_test = df.loc[test_mask, feature_cols].astype('float32')
y_test = df.loc[test_mask, 'net_demand'].astype('float32')

test_dates = df.loc[test_mask, 'date'].reset_index(drop=True)
test_hours = df.loc[test_mask, 'hour'].astype('int16').reset_index(drop=True)

print('Rows total:', len(df))
print('Train rows:', len(X_train))
print('Test rows:', len(X_test))
print('Cutoff date:', cutoff_date.date())


Rows total: 4266120
Train rows: 4184304
Test rows: 81816
Cutoff date: 2025-12-24


In [4]:
# Model: scale inputs + scale target for stable MLP optimization
import time

base_mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size=4096,
    learning_rate='adaptive',
    learning_rate_init=1e-3,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.05,
    n_iter_no_change=5,
    random_state=RANDOM_STATE,
    verbose=True,
)

model = TransformedTargetRegressor(
    regressor=Pipeline([
        ('x_scaler', StandardScaler(with_mean=True, with_std=True)),
        ('mlp', base_mlp),
    ]),
    transformer=StandardScaler(with_mean=True, with_std=True)
)

t0 = time.perf_counter()
model.fit(X_train, y_train)
train_seconds = time.perf_counter() - t0

pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
mae = float(mean_absolute_error(y_test, pred))
print(f'RMSE: {rmse:.4f}')
print(f'MAE : {mae:.4f}')
print(f'Training time: {train_seconds:.2f} sec ({train_seconds/60:.2f} min)')


Iteration 1, loss = 0.00721440
Validation score: 0.999667
Iteration 2, loss = 0.00011938
Validation score: 0.999833
Iteration 3, loss = 0.00007040
Validation score: 0.999884
Iteration 4, loss = 0.00005457
Validation score: 0.999925
Iteration 5, loss = 0.00005546
Validation score: 0.999952
Iteration 6, loss = 0.00005489
Validation score: 0.999955
Iteration 7, loss = 0.00006150
Validation score: 0.999968
Iteration 8, loss = 0.00003232
Validation score: 0.999961
Validation score did not improve more than tol=0.000100 for 5 consecutive epochs. Stopping.
RMSE: 1.0492
MAE : 0.6638
Training time: 158.70 sec (2.65 min)


In [5]:
# Hourly MAE breakdown (same reporting style as baseline)
eval_df = pd.DataFrame({
    'date': test_dates,
    'hour': test_hours,
    'y_true': y_test.values,
    'y_pred': pred,
})
hourly_mae = eval_df.groupby('hour').apply(
    lambda x: float(np.mean(np.abs(x['y_true'] - x['y_pred'])))
).reset_index(name='mae')

print(hourly_mae)


    hour       mae
0      0  0.701123
1      1  0.561512
2      2  0.527290
3      3  0.486379
4      4  0.587880
5      5  0.596003
6      6  0.636937
7      7  0.687216
8      8  0.782938
9      9  0.860601
10    10  0.749146
11    11  0.697335
12    12  0.633253
13    13  0.597023
14    14  0.611280
15    15  0.612914
16    16  0.664438
17    17  0.701062
18    18  0.727113
19    19  0.770675
20    20  0.740748
21    21  0.682202
22    22  0.621705
23    23  0.693570


/tmp/ipykernel_6775/3778856238.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hourly_mae = eval_df.groupby('hour').apply(


In [6]:
# Save artifacts
import joblib

model_path = ARTIFACT_DIR / 'mlp_model.joblib'
joblib.dump(model, model_path)

hourly_mae_path = ARTIFACT_DIR / 'hourly_mae.csv'
hourly_mae.to_csv(hourly_mae_path, index=False)

metrics = {
    'data_path': str(DATA_PATH),
    'model': 'MLPRegressor',
    'holdout_days': HOLDOUT_DAYS,
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'cutoff_date': str(cutoff_date),
    'mae': mae,
    'rmse': rmse,
    'feature_columns': feature_cols,
}

metrics_path = ARTIFACT_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved:')
print('-', model_path)
print('-', hourly_mae_path)
print('-', metrics_path)


Saved:
- /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/mlp/v1/mlp_model.joblib
- /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/mlp/v1/hourly_mae.csv
- /content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction/artifacts/model_training/mlp/v1/metrics.json
